Charger le fichier products.csv. En utilisant pyspark, repérer les valeurs aberrantes (ordre de grandeur : plusieurs centaines).

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab2Productpp") \
    .getOrCreate()

In [ ]:
from functools import reduce

from pyspark.sql.functions import coalesce, col, lit

In [12]:
product_df = spark.read.csv("../../data/products.csv", header=True, inferSchema=True)

In [13]:
product_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- code: double (nullable = true)
 |-- fat_100g: double (nullable = true)
 |-- saturated-fat_100g: double (nullable = true)
 |-- sugars_100g: double (nullable = true)
 |-- fiber_100g: double (nullable = true)
 |-- proteins_100g: double (nullable = true)
 |-- salt_100g: double (nullable = true)
 |-- sodium_100g: double (nullable = true)
 |-- autre: double (nullable = true)



In [36]:
filled_product_df = product_df.fillna(0)

## Valeurs négatives

In [37]:
col_to_check_neg = ["fat_100g", "saturated-fat_100g", "sugars_100g", "fiber_100g", "proteins_100g", "salt_100g", "sodium_100g", "autre"]

In [42]:
neg_condition = " OR ".join([f"`{c}` < 0" for c in col_to_check_neg])

filled_product_df.filter(neg_condition).toPandas()

,_c0,code,fat_100g,saturated-fat_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,autre
0,8582,1.121342e+10,0.00,0.00,-1.20,1.2,2.41,0.38354,0.151000,97.055460
1,18209,2.113049e+10,0.80,0.00,-0.80,0.8,0.80,0.87376,0.344000,97.182240
2,23784,2.840023e+10,33.33,13.33,0.00,-6.7,0.00,6.43382,2.533000,51.073180
3,33781,3.680042e+10,46.43,8.93,3.57,3.6,-3.57,0.99822,0.393000,39.648780
4,115310,4.029816e+06,0.00,0.00,0.00,0.0,-500.00,25.40000,10.000000,564.600000
5,117739,6.088670e+11,3.57,0.00,-3.57,3.6,7.14,0.95250,0.375000,87.932500
6,146284,7.892803e+11,13.33,3.33,-6.67,6.7,0.00,2.03200,0.800000,80.478000
7,150858,8.139220e+11,6.25,1.25,-6.25,1.2,1.25,1.19380,0.470000,94.636200
8,164030,8.563360e+11,21.43,3.57,-17.86,17.9,17.86,1.93294,0.761000,54.406060
9,169119,8.752080e+11,0.00,0.00,0.00,0.0,-800.00,7.62000,3.000000,889.380000


In [ ]:
negative_value = reduce(lambda a, b: a | b, (col(c) < 0 for c in col_to_check_neg))
negative = product_df.filter(negative_value)
negative.count()

11

In [ ]:
negative_value = reduce(lambda acc, new_value: acc | (col(new_value) < 0), col_to_check_neg, lit(False))
negative = product_df.filter(negative_value)
negative.count()

11

### Valeurs plus grandes que 100

In [56]:
col_to_check_more_100 = ["fat_100g", "saturated-fat_100g", "sugars_100g", "fiber_100g", "proteins_100g", "salt_100g", "sodium_100g"]

In [57]:
from functools import reduce
condition = reduce(lambda a, b: a | b, (col(c) > 100 for c in col_to_check_more_100))
more_than_100_df = product_df.filter(condition)
more_than_100_df.count()

176

### Fat et saturated fat 

In [60]:
fat_df = product_df.filter(col("saturated-fat_100g") > col("fat_100g"))
fat_df.count()

354

In [61]:
fat_df = filled_product_df.filter(col("saturated-fat_100g") > col("fat_100g"))
fat_df.count()

15668

### Sodium et salt

In [65]:
product_df.filter(col("sodium_100g") > col("salt_100g")).count()

0

In [75]:
M_Na = 23.0
M_Cl = 35.0

ratio = M_Na /( M_Na + M_Cl)
print(ratio)

tol = 0.02 # relatif

0.39655172413793105


In [76]:
condition = (
    (col("sodium_100g") < ratio * (1 - tol) * col("salt_100g"))
    |
    (col("sodium_100g") > ratio * (1 + tol) * col("salt_100g"))
)

product_df.filter(condition).count()

0

### Somme avec autre

In [ ]:
product_df = product_df.withColumn(
    "imputed_fat_100g",
    coalesce(col("fat_100g"), col("saturated-fat_100g"))
)

In [103]:
cols_to_sum = ["imputed_fat_100g", "sugars_100g", "fiber_100g", "proteins_100g", "salt_100g", "autre"]

In [ ]:
product_df = product_df.withColumn(
    "sum",
    sum(col(col_name) for col_name in cols_to_sum)
)

In [105]:
tol = 0.05

In [106]:
less_than_100_df = product_df.filter(col("sum") < 100 - tol)
more_than_100_df = product_df.filter(col("sum") > 100 + tol)

In [107]:
print(less_than_100_df.count())
print(more_than_100_df.count())

161620
1023


In [108]:
less_than_100_df.toPandas()

,_c0,code,fat_100g,saturated-fat_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g,autre,sum,imputed_fat_100g
0,1,4.530000e+03,28.57,28.57,14.29,3.6,3.57,0.00000,0.000000,21.400000,71.430000,28.57
1,2,4.559000e+03,17.86,0.00,17.86,7.1,17.86,0.63500,0.250000,38.435000,99.750000,17.86
2,3,1.608700e+04,57.14,5.36,3.57,7.1,17.86,1.22428,0.482000,7.263720,94.158000,57.14
3,7,1.612400e+04,18.75,4.69,15.62,9.4,14.06,0.13970,0.055000,37.285300,95.255000,18.75
4,12,1.687200e+04,36.67,5.00,3.33,6.7,16.67,1.60782,0.633000,29.389180,94.367000,36.67
...,...,...,...,...,...,...,...,...,...,...,...,...
161615,320740,9.782211e+12,NaN,12.00,10.50,0.0,8.70,0.29000,0.114173,68.395827,99.885827,12.00
161616,320741,9.782401e+12,NaN,1.00,1.00,10.0,10.00,10.00000,3.937008,64.062992,96.062992,1.00
161617,320751,9.847548e+12,2.80,0.60,2.60,5.9,13.00,0.68000,0.267717,74.152283,99.132283,2.80
161618,320756,9.898980e+05,31.00,NaN,9.60,1.1,2.10,1.10000,0.433071,54.666929,99.566929,31.00
